In [ ]:
import requests
import json
import os
import logging
from typing import Optional

# Profesyonel standartlarda loglama yapılandırması
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

OUTPUT_DIR = '../data/'

def fetch_ibb_data(resource_id: str, file_name: str) -> bool:
    """
    İBB Açık Veri Portalı'ndan belirtilen resource_id ile veriyi çeker 
    ve JSON formatında hedef klasöre kaydeder.
    
    Args:
        resource_id (str): İBB Portalındaki veri setinin benzersiz kimliği.
        file_name (str): Kaydedilecek dosyanın adı (uzantısız).
        
    Returns:
        bool: İşlem başarılıysa True, hata durumunda False döner.
    """
    logging.info("API istegi baslatiliyor: %s", file_name)
    
    url = f"https://data.ibb.gov.tr/api/3/action/datastore_search?resource_id={resource_id}&limit=15000"
    
    try:
        # timeout eklemek, sunucu yanıt vermediğinde scriptin sonsuza dek beklemesini önler
        response = requests.get(url, timeout=15)
        response.raise_for_status()  # HTTP 4xx ve 5xx hatalarını otomatik yakalar
        
        data = response.json()
        records = data.get('result', {}).get('records', [])
        
        if not records:
            logging.warning("API baglantisi basarili ancak kayit bulunamadi: %s", file_name)
            return False
            
        # Hedef klasör yoksa oluştur (hata almamak için)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        file_path = os.path.join(OUTPUT_DIR, f"{file_name}.json")
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, indent=4)
            
        logging.info("Basarili. Toplam %d kayit '%s.json' olarak kaydedildi.", len(records), file_name)
        return True
        
    except requests.exceptions.RequestException as req_err:
        logging.error("HTTP Istek Hatasi (%s): %s", file_name, req_err)
    except json.JSONDecodeError:
        logging.error("JSON Cozumleme Hatasi: API gecerli bir JSON dondurmedi (%s)", file_name)
    except Exception as e:
        logging.error("Beklenmeyen Hata (%s): %s", file_name, str(e))
        
    return False

if __name__ == "__main__":
    # İETT Durak Verisi
    DURAK_RESOURCE_ID = "21406852-c3f9-4467-8fa5-05e830e28151" 
    fetch_ibb_data(DURAK_RESOURCE_ID, "api_canli_duraklar")

    # İETT Hat Güzergahları Verisi
    HAT_RESOURCE_ID = "c3eb0d72-1ce4-4983-a3a8-6b0b4b19fcb9"
    fetch_ibb_data(HAT_RESOURCE_ID, "api_canli_hatlar")